# DRONE_TRACK : YOLOv8n 드론 탐지 학습 (Colab)

c01 데이터 변환 → c02 학습 → c05 ONNX 변환과 calib 준비까지 이 노트북 하나로 한다.

시작 전에 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택한다.
결과는 구글 드라이브 `MyDrive/DRONE_TRACK` 에 저장되므로 세션이 끊겨도 남는다.

## 0. 준비

In [ ]:
# GPU 확인 : Tesla T4 가 보이면 정상
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/DRONE_TRACK")   # 결과 보관 (세션이 끊겨도 남음)
WORK_DIR  = Path("/content/drone_det")                    # 학습 데이터 (빠른 로컬 디스크)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# 원본 category 번호 순서 그대로 사용
CLASS_NAMES = ["drone", "bird", "airplane", "helicopter"]

MIN_BOX_SIZE = 4   # 가로나 세로가 4픽셀보다 작은 박스는 버림 (학습에 방해)

## 1. c01 데이터 변환

Hugging Face `athkot/drone-vs-bird-synthetic-10k` (CC BY 4.0) 를 받아 YOLO 형식으로 바꾼다.

```
drone_det/
├── images/train/xxx.jpg   images/val/xxx.jpg
├── labels/train/xxx.txt   labels/val/xxx.txt   ← 한 줄에 박스 하나: 클래스 cx cy w h (0~1)
└── data.yaml
```

In [ ]:
from datasets import load_dataset

ds = load_dataset("athkot/drone-vs-bird-synthetic-10k")
print(ds)
print(ds["train"].features)

### 1-1. 박스 변환 함수  ✍️ 직접 입력

원본 박스는 `[x, y, w, h]` = 왼쪽 위 꼭짓점 좌표와 너비, 높이 (픽셀).
YOLO 는 `[cx, cy, w, h]` = 박스 **중심** 좌표와 너비, 높이를 **사진 크기로 나눈 0~1 값** 으로 받는다.

In [ ]:
def to_yolo(box, image_width, image_height):
    x, y, w, h = box

    # ✍️ 여기를 직접 입력

    return cx, cy, nw, nh


# 확인 : 1000x500 사진의 [100, 50, 200, 100] 박스 → (0.2, 0.2, 0.2, 0.2)
print(to_yolo([100, 50, 200, 100], 1000, 500))

In [ ]:
import io
from tqdm.auto import tqdm

def export_split(split_name, out_name):
    image_dir = WORK_DIR / "images" / out_name
    label_dir = WORK_DIR / "labels" / out_name
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    box_count = [0] * len(CLASS_NAMES)
    dropped = 0

    for row in tqdm(ds[split_name], desc=out_name):
        image_bytes = row["image"]["bytes"] if isinstance(row["image"], dict) else None

        stem = str(row["image_id"])
        width, height = row["width"], row["height"]

        # 사진 저장 (PIL Image 로 디코딩된 경우와 bytes 인 경우 모두 처리)
        if image_bytes is not None:
            ext = ".png" if image_bytes[:4] == b"\x89PNG" else ".jpg"
            (image_dir / f"{stem}{ext}").write_bytes(image_bytes)
        else:
            row["image"].convert("RGB").save(image_dir / f"{stem}.jpg", quality=95)

        # 라벨 저장 : 박스가 없는 사진은 빈 txt (= 배경 사진)
        lines = []
        for box, category in zip(row["objects"]["bbox"], row["objects"]["category"]):
            if box[2] < MIN_BOX_SIZE or box[3] < MIN_BOX_SIZE:
                dropped += 1
                continue
            cx, cy, nw, nh = to_yolo(box, width, height)
            lines.append(f"{category} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            box_count[category] += 1

        (label_dir / f"{stem}.txt").write_text("\n".join(lines))

    print(out_name, "박스 수:", dict(zip(CLASS_NAMES, box_count)), "버린 작은 박스:", dropped)


export_split("train", "train")
export_split("validation", "val")

In [ ]:
yaml_text = f"""path: {WORK_DIR}
train: images/train
val: images/val
names:
""" + "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASS_NAMES))

(WORK_DIR / "data.yaml").write_text(yaml_text)
print(yaml_text)

### 1-2. 라벨 확인 : 박스가 물체에 정확히 맞으면 변환 성공

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

label_files = [p for p in (WORK_DIR / "labels" / "train").glob("*.txt") if p.read_text().strip()]
colors = ["red", "lime", "cyan", "yellow"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, label_file in zip(axes.flat, random.sample(label_files, 6)):
    image_file = next((WORK_DIR / "images" / "train").glob(label_file.stem + ".*"))
    image = Image.open(image_file)
    W, H = image.size
    ax.imshow(image)
    for line in label_file.read_text().splitlines():
        c, cx, cy, w, h = line.split()
        c, cx, cy, w, h = int(c), float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        ax.add_patch(patches.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, color=colors[c], linewidth=2))
        ax.text(cx - w / 2, cy - h / 2 - 4, CLASS_NAMES[c], color=colors[c], fontsize=9)
    ax.set_title(image_file.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. c02 학습

### 2-1. 학습 실행  ✍️ 직접 입력

COCO 데이터로 미리 학습된 `yolov8n.pt` 를 불러와 드론 데이터로 추가 학습한다 (전이학습).
결과는 드라이브의 `DRIVE_DIR / "runs" / "yolov8n_drone"` 에 저장된다.

In [ ]:
from ultralytics import YOLO

# ✍️ 여기를 직접 입력

### 2-2. 결과 확인

In [ ]:
RUN_DIR = DRIVE_DIR / "runs" / "yolov8n_drone"

from IPython.display import Image as ShowImage, display
for name in ["results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"]:
    if (RUN_DIR / name).exists():
        print(name)
        display(ShowImage(str(RUN_DIR / name), width=900))

In [ ]:
# 검증 데이터로 최종 점수 (mAP50 : 박스가 50% 이상 겹치면 정답으로 친 점수)
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))
metrics = best.val(data=str(WORK_DIR / "data.yaml"), imgsz=640)
print("mAP50    :", round(metrics.box.map50, 4))
print("mAP50-95 :", round(metrics.box.map, 4))
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:10s} mAP50 = {metrics.box.ap50[i]:.4f}")

## 3. c05 ONNX 변환과 calib 준비

Hailo Model Zoo `yolov8n` 설정과 맞추기 위해 **640 고정, opset 11, NMS 제외** 로 변환한다.

In [ ]:
import shutil

onnx_path = best.export(format="onnx", imgsz=640, opset=11, simplify=True, dynamic=False)
shutil.copy(onnx_path, DRIVE_DIR / "yolov8n_drone.onnx")
print("저장:", DRIVE_DIR / "yolov8n_drone.onnx")

In [ ]:
import onnx
m = onnx.load(str(DRIVE_DIR / "yolov8n_drone.onnx"))
print("입력:", [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print("출력:", [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])

In [ ]:
# calib : 검증 사진 중 박스가 있는 사진 100장 (클래스가 고르게 섞이도록 무작위)
calib_dir = DRIVE_DIR / "calib"
calib_dir.mkdir(exist_ok=True)

random.seed(42)
val_labels = [p for p in (WORK_DIR / "labels" / "val").glob("*.txt") if p.read_text().strip()]
for label_file in random.sample(val_labels, 100):
    image_file = next((WORK_DIR / "images" / "val").glob(label_file.stem + ".*"))
    shutil.copy(image_file, calib_dir / image_file.name)

print("calib 사진:", len(list(calib_dir.iterdir())))

## 4. 맥으로 가져갈 것

드라이브 `MyDrive/DRONE_TRACK` 에서 아래를 내려받아 맥 `DRONE_TRACK/output`, `DRONE_TRACK/calib` 에 둔다.

1. `yolov8n_drone.onnx`
2. `calib/` 폴더 (100장)
3. `runs/yolov8n_drone/weights/best.pt` (보관용)